## Training a GP surrogate model

This notebook demonstrates how to train a GP model to represent the outputs of an ODE model.


In [ ]:
import numpy as np
import pandas as pd
import pickle
import torch

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

### Generate a training data set


In [ ]:
def analytical_model(dose, v, ka, cl, t):
    """Analytical expression of a 1 compartment PK model

    Args:
        t: time in h
        d: Dose in mg
        v: Distribution volume in mL
        ka: Absorption rate constant in mL/h
        cl: Clearance rate constant in mL/h

    Returns:
        y: Predicted concentration
    """
    ke = cl / v
    y1 = dose * ka / (v * (ka - ke)) * (torch.exp(-ke * t) - torch.exp(-ka * t)) + 1e-8
    return y1


variable_names = ["concentration"]

protocol_design = pd.DataFrame({"protocol_arm": ["arm-A", "arm-B"], "dose": [0.5, 2]})
nb_protocols = len(protocol_design)
struct_model = StructuralAnalytical(
    analytical_model, variable_names, protocol_design=protocol_design
)

In [ ]:
nb_timesteps = 15
tmax = 24.0
time_steps = np.linspace(0.0, tmax, nb_timesteps).tolist()

log_nb_patients = 5
param_ranges = {
    "ka": {"low": 0.02, "high": 0.07, "log": False},
    "cl": {"low": 0.1, "high": 0.3, "log": False},
    "v": {"low": -3, "high": 2, "log": True},
}

protocol_design = pd.DataFrame({"protocol_arm": ["A", "B"], "dose": [0.1, 1.0]})

In [ ]:
dataset = generate_training_data(
    struct_model=struct_model,
    ranges=param_ranges,
    log_nb_ind=log_nb_patients,
    time=time_steps,
)

In [ ]:
# Remove parts of the data set
bootstrapped_dataset = dataset.sample(frac=0.5)

### Define and train a GP model


In [ ]:
learned_ode_params = list(param_ranges.keys())
descriptors = learned_ode_params + ["time"]
print(descriptors)

# initiate our GP class
myGP = GP(
    bootstrapped_dataset,
    descriptors,
    var_strat="IMV",  # either IMV (Independent Multitask Variational) or LMCV (Linear Model of Coregionalization Variational)
    kernel="RBF",  # Either RBF or SMK
    nb_inducing_points=100,
    mll="ELBO",  # default, otherwise PLL
    nb_training_iter=1000,
    training_proportion=0.7,
    learning_rate=0.1,
    lr_decay=0.99,
    jitter=1e-6,
    log_inputs=learned_ode_params,
    log_outputs=["A0", "A1"],
    plot_frame_rate=50,
)

In [ ]:
myGP.train()

### Assess the goodness of fit of the GP


In [ ]:
myGP.eval_perf()

In [ ]:
myGP.plot_obs_vs_predicted(data_set="training")

In [ ]:
j = int(np.random.randint(0, dataset["id"].drop_duplicates().shape[0]))
myGP.plot_individual_solution(j)

In [ ]:
myGP.plot_all_solutions("training")

## Save your trained model in a pickle for later use


In [ ]:
model_file = "gp_model_example.pkl"
folder_path = "./"
with open(folder_path + model_file, "wb") as file:
    pickle.dump(myGP, file)
print(f"Model saved to {model_file}")